# Fitness Mirror - Unified YOLOv8 Nano + DNN Model

This notebook creates a **single unified model** that combines YOLOv8 Nano for pose keypoint extraction with additional DNN layers for:
- Exercise classification (what exercise is being performed)
- Form accuracy assessment (how accurate the form is)

The model is optimized for the M55M1 board and exports to ONNX and TFLite (int8) formats.

## Step 1: Install Required Libraries

In [ ]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
%pip install ultralytics
%pip install opencv-python
%pip install numpy
%pip install onnx
%pip install onnxruntime
%pip install tensorflow
%pip install tf-keras
%pip install pillow
%pip install matplotlib
%pip install scikit-learn

print("✓ All libraries installed successfully!")

## Step 2: Import Libraries and Configuration

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from ultralytics import YOLO
import cv2
import numpy as np
import os
import json
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image
import onnx
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Configuration
CONFIG = {
    'EXERCISES': ['hip thrusts', 'jumping jacks', 'lunges', 'pullups', 'pushups', 'squats'],
    'NUM_KEYPOINTS': 17,  # COCO format keypoints
    'KEYPOINT_DIM': 3,    # x, y, confidence
    'BATCH_SIZE': 16,
    'LEARNING_RATE': 0.001,
    'EPOCHS': 50,
    'DEVICE': 'cuda' if torch.cuda.is_available() else 'cpu',
    'DATA_DIR': r'c:\Users\Jiesheng He\OneDrive\Documents\aifitnessmirror\data\videos',
    'OUTPUT_DIR': r'c:\Users\Jiesheng He\OneDrive\Documents\aifitnessmirror\output\models',
    'FORM_ACCURACY_THRESHOLD': 0.7,
    
    # Display settings for video testing
    'DISPLAY': {
        'TEXT_COLOR': (0, 255, 0),  # Green (BGR format) - can be adjusted
        'TEXT_BG_COLOR': (0, 0, 0),  # Black background for text
        'FONT_SCALE': 1.0,
        'THICKNESS': 2,
        'PADDING': 10
    }
}

print(f"✓ Libraries imported successfully!")
print(f"✓ Using device: {CONFIG['DEVICE']}")
print(f"✓ Configured for {len(CONFIG['EXERCISES'])} exercises: {', '.join(CONFIG['EXERCISES'])}")

## Step 3: Create Unified Model Architecture

This model combines YOLOv8 Nano's pose detection backbone with custom DNN layers for:
1. **Exercise Classification**: Identifies which exercise is being performed
2. **Form Accuracy Assessment**: Evaluates how correctly the exercise is performed

The architecture is a **single unified model** that can be exported as one file.

In [ ]:
class UnifiedFitnessModel(nn.Module):
    """
    Unified model combining YOLOv8 Nano for keypoint extraction with custom DNN layers
    for exercise classification and form accuracy assessment.
    """
    def __init__(self, num_exercises=6, num_keypoints=17, keypoint_dim=3):
        super(UnifiedFitnessModel, self).__init__()
        
        # Load YOLOv8 Nano pose model
        self.yolo_model = YOLO('yolov8n-pose.pt')
        
        # Freeze YOLO backbone initially (can unfreeze later for fine-tuning)
        for param in self.yolo_model.model.parameters():
            param.requires_grad = False
        
        # Calculate input size for DNN layers
        self.input_features = num_keypoints * keypoint_dim  # 17 * 3 = 51
        
        # Shared feature extraction layers
        self.shared_layers = nn.Sequential(
            nn.Linear(self.input_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        # Exercise classification head
        self.exercise_classifier = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, num_exercises)
        )
        
        # Form accuracy assessment head
        self.form_accuracy_head = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1),
            nn.Sigmoid()  # Output between 0-1 for accuracy percentage
        )
        
        self.num_exercises = num_exercises
        self.num_keypoints = num_keypoints
        self.keypoint_dim = keypoint_dim
        
    def extract_keypoints(self, image):
        """
        Extract keypoints from image using YOLO model
        Returns: tensor of shape (batch_size, num_keypoints * keypoint_dim)
        """
        # Run YOLO inference
        results = self.yolo_model(image, verbose=False)
        
        keypoints_batch = []
        for result in results:
            if result.keypoints is not None and len(result.keypoints.data) > 0:
                # Get first person's keypoints (primary subject)
                kpts = result.keypoints.data[0].cpu()  # Shape: (17, 3)
                keypoints_flat = kpts.flatten()  # Shape: (51,)
            else:
                # No person detected, use zeros
                keypoints_flat = torch.zeros(self.input_features)
            
            keypoints_batch.append(keypoints_flat)
        
        return torch.stack(keypoints_batch).to(CONFIG['DEVICE'])
    
    def forward(self, x, return_keypoints=False):
        """
        Forward pass through the unified model
        Args:
            x: Input image or pre-extracted keypoints
            return_keypoints: If True, also return extracted keypoints
        Returns:
            exercise_logits: Classification logits for exercises
            form_accuracy: Form accuracy score (0-1)
            keypoints (optional): Extracted keypoints
        """
        # Check if input is image or keypoints
        if len(x.shape) == 4:  # Image input (batch, channels, height, width)
            keypoints = self.extract_keypoints(x)
        else:  # Pre-extracted keypoints
            keypoints = x
        
        # Ensure keypoints is 2D
        if len(keypoints.shape) == 1:
            keypoints = keypoints.unsqueeze(0)
        
        # Shared feature extraction
        shared_features = self.shared_layers(keypoints)
        
        # Exercise classification
        exercise_logits = self.exercise_classifier(shared_features)
        
        # Form accuracy assessment
        form_accuracy = self.form_accuracy_head(shared_features)
        
        if return_keypoints:
            return exercise_logits, form_accuracy, keypoints
        
        return exercise_logits, form_accuracy
    
    def predict(self, image):
        """
        Convenience method for inference
        Returns: exercise_name, form_accuracy, keypoints
        """
        self.eval()
        with torch.no_grad():
            exercise_logits, form_accuracy, keypoints = self.forward(image, return_keypoints=True)
            exercise_idx = torch.argmax(exercise_logits, dim=1).item()
            form_score = form_accuracy.item()
            
        return exercise_idx, form_score, keypoints

# Initialize the model
model = UnifiedFitnessModel(
    num_exercises=len(CONFIG['EXERCISES']),
    num_keypoints=CONFIG['NUM_KEYPOINTS'],
    keypoint_dim=CONFIG['KEYPOINT_DIM']
).to(CONFIG['DEVICE'])

print(f"✓ Unified Fitness Model created successfully!")
print(f"✓ Model structure:")
print(f"  - YOLOv8 Nano backbone for keypoint extraction")
print(f"  - Shared feature layers: 51 → 256 → 128 → 64")
print(f"  - Exercise classifier: 64 → 32 → {len(CONFIG['EXERCISES'])}")
print(f"  - Form accuracy head: 64 → 32 → 1")
print(f"✓ Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"✓ Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## Step 4: Data Preparation and Dataset Class

In [ ]:
class FitnessDataset(Dataset):
    """
    Dataset class for loading video frames and extracting keypoints
    """
    def __init__(self, video_paths, labels, form_scores, model, transform=None, frames_per_video=30):
        self.video_paths = video_paths
        self.labels = labels
        self.form_scores = form_scores
        self.model = model
        self.transform = transform
        self.frames_per_video = frames_per_video
        
        # Extract keypoints for all videos
        print("Extracting keypoints from videos...")
        self.keypoints_data = []
        self.labels_data = []
        self.form_scores_data = []
        
        for idx, video_path in enumerate(video_paths):
            keypoints_list = self._extract_keypoints_from_video(video_path)
            
            for kpts in keypoints_list:
                self.keypoints_data.append(kpts)
                self.labels_data.append(labels[idx])
                self.form_scores_data.append(form_scores[idx])
        
        print(f"✓ Extracted {len(self.keypoints_data)} frames from {len(video_paths)} videos")
    
    def _extract_keypoints_from_video(self, video_path):
        """Extract keypoints from video frames"""
        cap = cv2.VideoCapture(video_path)
        keypoints_list = []
        frame_count = 0
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        # Sample frames evenly
        frame_indices = np.linspace(0, total_frames - 1, self.frames_per_video, dtype=int)
        
        for frame_idx in frame_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            ret, frame = cap.read()
            
            if not ret:
                break
            
            # Run YOLO to extract keypoints
            results = self.model.yolo_model(frame, verbose=False)
            
            if results[0].keypoints is not None and len(results[0].keypoints.data) > 0:
                kpts = results[0].keypoints.data[0].cpu().numpy()  # Shape: (17, 3)
                keypoints_list.append(kpts.flatten())  # Shape: (51,)
            else:
                # No person detected
                keypoints_list.append(np.zeros(51))
        
        cap.release()
        return keypoints_list
    
    def __len__(self):
        return len(self.keypoints_data)
    
    def __getitem__(self, idx):
        keypoints = torch.FloatTensor(self.keypoints_data[idx])
        label = torch.LongTensor([self.labels_data[idx]])[0]
        form_score = torch.FloatTensor([self.form_scores_data[idx]])[0]
        
        return keypoints, label, form_score

def load_training_data():
    """
    Load video data from the data directory
    Returns: video_paths, labels, form_scores
    """
    video_paths = []
    labels = []
    form_scores = []
    
    label_encoder = LabelEncoder()
    label_encoder.fit(CONFIG['EXERCISES'])
    
    for exercise_idx, exercise in enumerate(CONFIG['EXERCISES']):
        exercise_dir = os.path.join(CONFIG['DATA_DIR'], exercise)
        
        if not os.path.exists(exercise_dir):
            print(f"⚠ Warning: Directory not found: {exercise_dir}")
            continue
        
        video_files = [f for f in os.listdir(exercise_dir) if f.endswith(('.mp4', '.avi', '.mov'))]
        
        for video_file in video_files:
            video_path = os.path.join(exercise_dir, video_file)
            video_paths.append(video_path)
            labels.append(exercise_idx)
            
            # For now, assign random form scores (in real scenario, these would be labeled)
            # You can replace this with actual labeled data
            form_scores.append(np.random.uniform(0.5, 1.0))
        
        print(f"✓ Loaded {len(video_files)} videos for '{exercise}'")
    
    return video_paths, labels, form_scores, label_encoder

# Load data
video_paths, labels, form_scores, label_encoder = load_training_data()

print(f"\n✓ Total videos loaded: {len(video_paths)}")
print(f"✓ Exercise distribution:")
for i, exercise in enumerate(CONFIG['EXERCISES']):
    count = labels.count(i)
    print(f"  - {exercise}: {count} videos")

In [ ]:
# Create train/validation split
if len(video_paths) > 0:
    train_videos, val_videos, train_labels, val_labels, train_form, val_form = train_test_split(
        video_paths, labels, form_scores, test_size=0.2, random_state=42, stratify=labels
    )
    
    # Create datasets
    print("\nCreating training dataset...")
    train_dataset = FitnessDataset(train_videos, train_labels, train_form, model, frames_per_video=30)
    
    print("\nCreating validation dataset...")
    val_dataset = FitnessDataset(val_videos, val_labels, val_form, model, frames_per_video=15)
    
    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=CONFIG['BATCH_SIZE'], shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=CONFIG['BATCH_SIZE'], shuffle=False)
    
    print(f"\n✓ Training samples: {len(train_dataset)}")
    print(f"✓ Validation samples: {len(val_dataset)}")
    print(f"✓ Batch size: {CONFIG['BATCH_SIZE']}")
    print(f"✓ Training batches: {len(train_loader)}")
    print(f"✓ Validation batches: {len(val_loader)}")
else:
    print("⚠ No training data found. Please add videos to the data/videos directory.")
    train_loader = None
    val_loader = None

## Step 5: Training Loop

In [ ]:
def train_model(model, train_loader, val_loader, epochs=50):
    """
    Train the unified fitness model
    """
    # Loss functions
    criterion_classification = nn.CrossEntropyLoss()
    criterion_regression = nn.MSELoss()
    
    # Optimizer
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=CONFIG['LEARNING_RATE']
    )
    
    # Learning rate scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, verbose=True
    )
    
    # Training history
    history = {
        'train_loss': [],
        'val_loss': [],
        'train_acc': [],
        'val_acc': [],
        'train_form_mae': [],
        'val_form_mae': []
    }
    
    best_val_loss = float('inf')
    
    print(f"\n{'='*60}")
    print(f"Starting training for {epochs} epochs")
    print(f"{'='*60}\n")
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        train_form_error = 0.0
        
        for batch_idx, (keypoints, labels, form_scores) in enumerate(train_loader):
            keypoints = keypoints.to(CONFIG['DEVICE'])
            labels = labels.to(CONFIG['DEVICE'])
            form_scores = form_scores.to(CONFIG['DEVICE'])
            
            # Forward pass
            exercise_logits, form_pred = model(keypoints)
            
            # Calculate losses
            loss_classification = criterion_classification(exercise_logits, labels)
            loss_regression = criterion_regression(form_pred.squeeze(), form_scores)
            
            # Combined loss (you can adjust weights)
            loss = loss_classification + 0.5 * loss_regression
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            # Statistics
            train_loss += loss.item()
            _, predicted = torch.max(exercise_logits, 1)
            train_correct += (predicted == labels).sum().item()
            train_total += labels.size(0)
            train_form_error += torch.abs(form_pred.squeeze() - form_scores).sum().item()
        
        # Calculate training metrics
        avg_train_loss = train_loss / len(train_loader)
        train_accuracy = 100.0 * train_correct / train_total
        train_form_mae = train_form_error / train_total
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        val_form_error = 0.0
        
        with torch.no_grad():
            for keypoints, labels, form_scores in val_loader:
                keypoints = keypoints.to(CONFIG['DEVICE'])
                labels = labels.to(CONFIG['DEVICE'])
                form_scores = form_scores.to(CONFIG['DEVICE'])
                
                # Forward pass
                exercise_logits, form_pred = model(keypoints)
                
                # Calculate losses
                loss_classification = criterion_classification(exercise_logits, labels)
                loss_regression = criterion_regression(form_pred.squeeze(), form_scores)
                loss = loss_classification + 0.5 * loss_regression
                
                # Statistics
                val_loss += loss.item()
                _, predicted = torch.max(exercise_logits, 1)
                val_correct += (predicted == labels).sum().item()
                val_total += labels.size(0)
                val_form_error += torch.abs(form_pred.squeeze() - form_scores).sum().item()
        
        # Calculate validation metrics
        avg_val_loss = val_loss / len(val_loader)
        val_accuracy = 100.0 * val_correct / val_total
        val_form_mae = val_form_error / val_total
        
        # Update history
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['train_acc'].append(train_accuracy)
        history['val_acc'].append(val_accuracy)
        history['train_form_mae'].append(train_form_mae)
        history['val_form_mae'].append(val_form_mae)
        
        # Learning rate scheduling
        scheduler.step(avg_val_loss)
        
        # Print progress
        print(f"Epoch [{epoch+1}/{epochs}]")
        print(f"  Train Loss: {avg_train_loss:.4f} | Train Acc: {train_accuracy:.2f}% | Form MAE: {train_form_mae:.4f}")
        print(f"  Val Loss:   {avg_val_loss:.4f} | Val Acc:   {val_accuracy:.2f}% | Form MAE: {val_form_mae:.4f}")
        
        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': avg_val_loss,
                'val_accuracy': val_accuracy,
            }, os.path.join(CONFIG['OUTPUT_DIR'], 'best_model.pth'))
            print(f"  ✓ Best model saved!")
        
        print()
    
    print(f"\n{'='*60}")
    print(f"Training completed!")
    print(f"Best validation loss: {best_val_loss:.4f}")
    print(f"{'='*60}\n")
    
    return history

# Train the model (only if data is available)
if train_loader is not None and val_loader is not None:
    history = train_model(model, train_loader, val_loader, epochs=CONFIG['EPOCHS'])
    
    # Plot training history
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Loss plot
    axes[0].plot(history['train_loss'], label='Train Loss')
    axes[0].plot(history['val_loss'], label='Val Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training and Validation Loss')
    axes[0].legend()
    axes[0].grid(True)
    
    # Accuracy plot
    axes[1].plot(history['train_acc'], label='Train Accuracy')
    axes[1].plot(history['val_acc'], label='Val Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].set_title('Training and Validation Accuracy')
    axes[1].legend()
    axes[1].grid(True)
    
    # Form MAE plot
    axes[2].plot(history['train_form_mae'], label='Train Form MAE')
    axes[2].plot(history['val_form_mae'], label='Val Form MAE')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Mean Absolute Error')
    axes[2].set_title('Form Accuracy MAE')
    axes[2].legend()
    axes[2].grid(True)
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠ Skipping training - no data available")

## Step 6: Video Testing with Dynamic Display

Test the model on MP4 videos with:
- Real-time exercise classification
- Form accuracy assessment
- Dynamic text sizing based on video dimensions
- Adjustable text color to avoid blending with background

In [ ]:
def calculate_dynamic_display_params(frame_height, frame_width):
    """
    Calculate dynamic display parameters based on video dimensions
    """
    # Base font scale on video height
    base_font_scale = frame_height / 720.0  # 720p as baseline
    font_scale = max(0.5, min(2.0, base_font_scale))  # Clamp between 0.5 and 2.0
    
    # Thickness proportional to font scale
    thickness = max(1, int(2 * font_scale))
    
    # Padding proportional to frame size
    padding = int(10 * font_scale)
    
    return font_scale, thickness, padding

def get_contrasting_color(frame, position):
    """
    Get a contrasting color based on the background at the given position
    """
    y, x = position
    h, w = frame.shape[:2]
    
    # Sample area around position
    sample_y = max(0, min(h-1, y))
    sample_x = max(0, min(w-1, x))
    
    # Get average color in small region
    region_size = 20
    y1 = max(0, sample_y - region_size)
    y2 = min(h, sample_y + region_size)
    x1 = max(0, sample_x - region_size)
    x2 = min(w, sample_x + region_size)
    
    region = frame[y1:y2, x1:x2]
    avg_color = region.mean(axis=(0, 1))
    
    # Calculate brightness
    brightness = 0.299 * avg_color[2] + 0.587 * avg_color[1] + 0.114 * avg_color[0]
    
    # Return contrasting color
    if brightness > 127:
        return (0, 0, 0)  # Black for bright backgrounds
    else:
        return (255, 255, 255)  # White for dark backgrounds

def draw_text_with_background(frame, text, position, font_scale, thickness, text_color, bg_color, padding):
    """
    Draw text with a background rectangle for better visibility
    """
    font = cv2.FONT_HERSHEY_SIMPLEX
    
    # Get text size
    (text_width, text_height), baseline = cv2.getTextSize(text, font, font_scale, thickness)
    
    x, y = position
    
    # Draw background rectangle
    cv2.rectangle(
        frame,
        (x - padding, y - text_height - padding),
        (x + text_width + padding, y + baseline + padding),
        bg_color,
        -1
    )
    
    # Draw text
    cv2.putText(
        frame,
        text,
        (x, y),
        font,
        font_scale,
        text_color,
        thickness,
        cv2.LINE_AA
    )
    
    return text_height + baseline + 2 * padding

def test_on_video(model, video_path, display_color=None, save_output=False):
    """
    Test the model on a video and display results in notebook
    
    Args:
        model: The trained model
        video_path: Path to the video file
        display_color: Custom color for text (BGR format), None for auto-contrast
        save_output: Whether to save output video
    """
    # Load best model if available
    model_path = os.path.join(CONFIG['OUTPUT_DIR'], 'best_model.pth')
    if os.path.exists(model_path):
        checkpoint = torch.load(model_path, map_location=CONFIG['DEVICE'])
        model.load_state_dict(checkpoint['model_state_dict'])
        print("✓ Loaded best model checkpoint")
    
    model.eval()
    
    # Open video
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"❌ Error: Could not open video {video_path}")
        return
    
    # Get video properties
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    print(f"\n{'='*60}")
    print(f"Testing on video: {os.path.basename(video_path)}")
    print(f"Resolution: {frame_width}x{frame_height}")
    print(f"FPS: {fps}")
    print(f"Total frames: {total_frames}")
    print(f"{'='*60}\n")
    
    # Calculate dynamic display parameters
    font_scale, thickness, padding = calculate_dynamic_display_params(frame_height, frame_width)
    
    # Prepare for saving frames
    frames_to_display = []
    frame_count = 0
    display_interval = max(1, fps // 2)  # Show 2 frames per second
    
    with torch.no_grad():
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            frame_count += 1
            
            # Process every frame but only save some for display
            original_frame = frame.copy()
            
            # Get predictions
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            exercise_idx, form_score, keypoints = model.predict(frame_rgb)
            
            exercise_name = CONFIG['EXERCISES'][exercise_idx]
            
            # Determine text color
            if display_color is None:
                text_color = get_contrasting_color(frame, (50, 50))
            else:
                text_color = display_color
            
            bg_color = CONFIG['DISPLAY']['TEXT_BG_COLOR']
            
            # Draw exercise name
            y_offset = 40
            exercise_text = f"Exercise: {exercise_name}"
            y_offset += draw_text_with_background(
                frame, exercise_text, (20, y_offset),
                font_scale, thickness, text_color, bg_color, padding
            )
            
            # Draw form accuracy
            form_percentage = form_score * 100
            form_text = f"Form Accuracy: {form_percentage:.1f}%"
            
            # Color code form accuracy
            if form_percentage >= 80:
                form_color = (0, 255, 0)  # Green - Good
            elif form_percentage >= 60:
                form_color = (0, 255, 255)  # Yellow - Fair
            else:
                form_color = (0, 0, 255)  # Red - Poor
            
            y_offset += draw_text_with_background(
                frame, form_text, (20, y_offset),
                font_scale, thickness, form_color, bg_color, padding
            )
            
            # Draw keypoints on frame
            results = model.yolo_model(original_frame, verbose=False)
            if results[0].keypoints is not None and len(results[0].keypoints.data) > 0:
                annotated_frame = results[0].plot()
                # Overlay text on annotated frame
                frame = annotated_frame
                
                # Redraw text on annotated frame
                y_offset = 40
                y_offset += draw_text_with_background(
                    frame, exercise_text, (20, y_offset),
                    font_scale, thickness, text_color, bg_color, padding
                )
                y_offset += draw_text_with_background(
                    frame, form_text, (20, y_offset),
                    font_scale, thickness, form_color, bg_color, padding
                )
            
            # Save frames for display (subsample to avoid too many frames)
            if frame_count % display_interval == 0:
                frames_to_display.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            
            # Progress update
            if frame_count % 30 == 0:
                progress = (frame_count / total_frames) * 100
                print(f"Processing: {progress:.1f}% ({frame_count}/{total_frames} frames)", end='\r')
    
    cap.release()
    print(f"\n✓ Processing complete!")
    
    # Display sample frames
    print(f"\nDisplaying {len(frames_to_display)} sample frames from the video:")
    
    # Create grid of frames
    num_frames = min(12, len(frames_to_display))  # Show up to 12 frames
    frames_to_show = frames_to_display[::max(1, len(frames_to_display)//num_frames)][:num_frames]
    
    cols = 4
    rows = (num_frames + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(20, 5*rows))
    axes = axes.flatten() if num_frames > 1 else [axes]
    
    for idx, frame in enumerate(frames_to_show):
        axes[idx].imshow(frame)
        axes[idx].axis('off')
        axes[idx].set_title(f'Frame {idx * display_interval}')
    
    # Hide unused subplots
    for idx in range(num_frames, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n✓ Video testing complete!")

# Example: Test on a video
# Uncomment and modify the path to test on your video
# test_on_video(model, r'path\to\your\video.mp4', display_color=None)

print("✓ Video testing function ready!")
print("\nTo test on a video, use:")
print("  test_on_video(model, r'path\\to\\video.mp4')")
print("\nOptional parameters:")
print("  - display_color=(B, G, R): Set custom text color, e.g., (0, 255, 0) for green")
print("  - display_color=None: Auto-detect contrasting color based on background")

## Step 7: Export Model to ONNX Format

Export the trained model as a single ONNX file for deployment on the M55M1 board.

In [ ]:
class ONNXExportWrapper(nn.Module):
    """
    Wrapper for ONNX export that takes image input and outputs predictions
    """
    def __init__(self, fitness_model):
        super(ONNXExportWrapper, self).__init__()
        self.model = fitness_model
        
    def forward(self, keypoints):
        """
        Forward pass for ONNX export
        Input: keypoints tensor of shape (batch, 51)
        Output: exercise_logits (batch, num_exercises), form_accuracy (batch, 1)
        """
        exercise_logits, form_accuracy = self.model(keypoints)
        return exercise_logits, form_accuracy

def export_to_onnx(model, output_path):
    """
    Export the unified model to ONNX format
    """
    print(f"\n{'='*60}")
    print("Exporting model to ONNX format")
    print(f"{'='*60}\n")
    
    # Load best model if available
    model_path = os.path.join(CONFIG['OUTPUT_DIR'], 'best_model.pth')
    if os.path.exists(model_path):
        checkpoint = torch.load(model_path, map_location=CONFIG['DEVICE'])
        model.load_state_dict(checkpoint['model_state_dict'])
        print("✓ Loaded best model checkpoint")
    
    model.eval()
    
    # Create export wrapper
    export_model = ONNXExportWrapper(model)
    export_model.eval()
    
    # Create dummy input (keypoints)
    dummy_input = torch.randn(1, CONFIG['NUM_KEYPOINTS'] * CONFIG['KEYPOINT_DIM']).to(CONFIG['DEVICE'])
    
    # Export to ONNX
    torch.onnx.export(
        export_model,
        dummy_input,
        output_path,
        export_params=True,
        opset_version=11,
        do_constant_folding=True,
        input_names=['keypoints'],
        output_names=['exercise_logits', 'form_accuracy'],
        dynamic_axes={
            'keypoints': {0: 'batch_size'},
            'exercise_logits': {0: 'batch_size'},
            'form_accuracy': {0: 'batch_size'}
        }
    )
    
    print(f"✓ Model exported to: {output_path}")
    
    # Verify the ONNX model
    onnx_model = onnx.load(output_path)
    onnx.checker.check_model(onnx_model)
    print("✓ ONNX model verification successful!")
    
    # Print model info
    file_size = os.path.getsize(output_path) / (1024 * 1024)  # Size in MB
    print(f"✓ Model size: {file_size:.2f} MB")
    
    # Save metadata
    metadata = {
        'model_type': 'Unified Fitness Model (YOLOv8 Nano + DNN)',
        'exercises': CONFIG['EXERCISES'],
        'num_keypoints': CONFIG['NUM_KEYPOINTS'],
        'keypoint_dim': CONFIG['KEYPOINT_DIM'],
        'input_shape': [1, CONFIG['NUM_KEYPOINTS'] * CONFIG['KEYPOINT_DIM']],
        'output_shapes': {
            'exercise_logits': [1, len(CONFIG['EXERCISES'])],
            'form_accuracy': [1, 1]
        },
        'export_date': str(np.datetime64('now'))
    }
    
    metadata_path = output_path.replace('.onnx', '_metadata.json')
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=4)
    
    print(f"✓ Metadata saved to: {metadata_path}")
    print(f"\n{'='*60}")
    
    return output_path

# Export the model
onnx_output_path = os.path.join(CONFIG['OUTPUT_DIR'], 'fitness_model_unified.onnx')
os.makedirs(CONFIG['OUTPUT_DIR'], exist_ok=True)

try:
    exported_path = export_to_onnx(model, onnx_output_path)
    print(f"\n✅ ONNX export successful!")
    print(f"📁 Saved to: {exported_path}")
except Exception as e:
    print(f"\n❌ Error during ONNX export: {str(e)}")
    print("This is normal if the model hasn't been trained yet.")

## Step 8: Convert to TFLite with INT8 Quantization

Convert the ONNX model to TFLite format with INT8 quantization for optimal performance on the M55M1 board.

In [ ]:
import onnx
from onnx_tf.backend import prepare
import tensorflow as tf

def onnx_to_tflite_int8(onnx_path, output_path, representative_dataset=None):
    """
    Convert ONNX model to TFLite with INT8 quantization
    
    Args:
        onnx_path: Path to ONNX model
        output_path: Path to save TFLite model
        representative_dataset: Generator function for calibration data
    """
    print(f"\n{'='*60}")
    print("Converting ONNX to TFLite with INT8 Quantization")
    print(f"{'='*60}\n")
    
    # Check if ONNX file exists
    if not os.path.exists(onnx_path):
        print(f"❌ ONNX file not found: {onnx_path}")
        print("Please run the ONNX export cell first.")
        return None
    
    try:
        # Load ONNX model
        print("Step 1: Loading ONNX model...")
        onnx_model = onnx.load(onnx_path)
        print("✓ ONNX model loaded successfully")
        
        # Convert ONNX to TensorFlow
        print("\nStep 2: Converting ONNX to TensorFlow...")
        tf_rep = prepare(onnx_model)
        
        # Export to SavedModel format
        savedmodel_path = onnx_path.replace('.onnx', '_savedmodel')
        tf_rep.export_graph(savedmodel_path)
        print(f"✓ TensorFlow SavedModel created: {savedmodel_path}")
        
        # Convert to TFLite with INT8 quantization
        print("\nStep 3: Converting to TFLite with INT8 quantization...")
        converter = tf.lite.TFLiteConverter.from_saved_model(savedmodel_path)
        
        # Enable INT8 quantization
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        
        # Set representative dataset for full integer quantization
        if representative_dataset is None:
            # Create a default representative dataset
            def default_representative_dataset():
                for _ in range(100):
                    # Generate random keypoints data
                    data = np.random.randn(1, CONFIG['NUM_KEYPOINTS'] * CONFIG['KEYPOINT_DIM']).astype(np.float32)
                    yield [data]
            
            representative_dataset = default_representative_dataset
        
        converter.representative_dataset = representative_dataset
        
        # Ensure input and output are int8
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8
        
        # Convert the model
        tflite_model = converter.convert()
        
        # Save the TFLite model
        with open(output_path, 'wb') as f:
            f.write(tflite_model)
        
        print(f"✓ TFLite model saved: {output_path}")
        
        # Print model info
        file_size = os.path.getsize(output_path) / 1024  # Size in KB
        print(f"✓ TFLite model size: {file_size:.2f} KB")
        
        # Verify the model
        print("\nStep 4: Verifying TFLite model...")
        interpreter = tf.lite.Interpreter(model_path=output_path)
        interpreter.allocate_tensors()
        
        input_details = interpreter.get_input_details()
        output_details = interpreter.get_output_details()
        
        print("\n📊 Model Information:")
        print(f"  Input details:")
        for detail in input_details:
            print(f"    - Name: {detail['name']}")
            print(f"      Shape: {detail['shape']}")
            print(f"      Type: {detail['dtype']}")
        
        print(f"\n  Output details:")
        for detail in output_details:
            print(f"    - Name: {detail['name']}")
            print(f"      Shape: {detail['shape']}")
            print(f"      Type: {detail['dtype']}")
        
        # Save conversion metadata
        metadata = {
            'source_onnx': onnx_path,
            'tflite_output': output_path,
            'quantization': 'INT8',
            'model_size_kb': file_size,
            'input_details': [
                {
                    'name': detail['name'],
                    'shape': detail['shape'].tolist(),
                    'dtype': str(detail['dtype'])
                }
                for detail in input_details
            ],
            'output_details': [
                {
                    'name': detail['name'],
                    'shape': detail['shape'].tolist(),
                    'dtype': str(detail['dtype'])
                }
                for detail in output_details
            ],
            'conversion_date': str(np.datetime64('now'))
        }
        
        metadata_path = output_path.replace('.tflite', '_metadata.json')
        with open(metadata_path, 'w') as f:
            json.dump(metadata, f, indent=4)
        
        print(f"\n✓ Metadata saved to: {metadata_path}")
        print(f"\n{'='*60}")
        print("✅ Conversion to TFLite INT8 successful!")
        print(f"{'='*60}\n")
        
        return output_path
        
    except Exception as e:
        print(f"\n❌ Error during conversion: {str(e)}")
        print("\nNote: If you see 'No module named onnx_tf', install it with:")
        print("  !pip install onnx-tf")
        return None

# Convert to TFLite
tflite_output_path = os.path.join(CONFIG['OUTPUT_DIR'], 'fitness_model_unified_int8.tflite')

try:
    # First install onnx-tf if not already installed
    print("Installing onnx-tf...")
    import subprocess
    subprocess.check_call(['pip', 'install', 'onnx-tf', '-q'])
    print("✓ onnx-tf installed\n")
    
    tflite_path = onnx_to_tflite_int8(onnx_output_path, tflite_output_path)
    
    if tflite_path:
        print(f"\n🎉 Model ready for deployment on M55M1 board!")
        print(f"\n📁 Files created:")
        print(f"  - ONNX model: {onnx_output_path}")
        print(f"  - TFLite INT8 model: {tflite_path}")
        print(f"  - Metadata files in the same directory")
except Exception as e:
    print(f"❌ Error: {str(e)}")
    print("This is normal if the model hasn't been trained yet.")